In [1]:
# Ajustes principais:
# - Para cada momento, definimos uma meta de recall mínimo da minoria.
# - Selecionamos o threshold que satisfaz a meta e maximiza a precisão da minoria (ou F1).
# - Mantemos tuning de max_depth por momento com base em f1_minority (pode mudar para recall_minority em momentos iniciais).

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix

CSV_PATH = r'dados\dados_modelo.csv'
TARGET = 'Aprovou_Aprovou'
random_state_global = 42
candidate_depths = [None, 4, 8, 12, 16]
n_estimators = 500
minority_label = 0.0

# Momentos (os seus)
M1 = ['Idade_Maior', 'Genero_Masculino', 'STEM_SI']
M2 = M1 + ['Quiz1', 'Quiz2', 'TempoQ1', 'TempoQ2']
M3 = M2 + ['Quiz3', 'Quiz4', 'TempoQ3', 'TempoQ4']
M4 = M3 + ['Quiz5', 'Quiz6', 'TempoQ5', 'TempoQ6', 'Oficinas']
M5 = M4 + ['Quiz7', 'TempoQ7', 'CalcNotaQuiz_scaled', 'MelhoraNotaQuizzes_True']
MOMENTS = {1:M1, 2:M2, 3:M3, 4:M4, 5:M5}

# Metas de recall mínimo por momento (ajuste conforme sua política)
RECALL_MIN_TARGET = {
    1: 0.50,
    2: 0.45,
    3: 0.40,
    4: 0.50,
    5: 0.60
}

def build_preprocess(X):
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()
    return ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
            ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
        ],
        remainder='drop'
    )

def make_pipe(X_subset, max_depth=None):
    preprocess = build_preprocess(X_subset)
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state_global,
        n_jobs=-1,
        class_weight='balanced'
    )
    return Pipeline([('prep', preprocess), ('rf', rf)])

def tune_depth_for_moment(X_train, y_train, X_val, y_val, cols, depths):
    best, best_f1, best_depth = None, -np.inf, None
    for md in depths:
        pipe = make_pipe(X_train[cols], max_depth=md)
        pipe.fit(X_train[cols], y_train)
        y_val_pred = pipe.predict(X_val[cols])
        f1_min = f1_score(y_val, y_val_pred, pos_label=minority_label, zero_division=0)
        if f1_min > best_f1:
            best, best_f1, best_depth = pipe, f1_min, md
    return best, best_depth, best_f1

def predict_with_threshold(pipe, X, threshold):
    classes_ = pipe.named_steps['rf'].classes_
    idx_min = list(classes_).index(minority_label)
    proba_min = pipe.predict_proba(X)[:, idx_min]
    return np.where(proba_min >= threshold, minority_label, 1.0)

def choose_threshold_with_recall_target(pipe, X_val, y_val, recall_target, preference='precision'):
    # preference: 'precision' (maximiza precisão da minoria) ou 'f1' entre thresholds que atendem recall_target
    classes_ = pipe.named_steps['rf'].classes_
    idx_min = list(classes_).index(minority_label)
    proba_min = pipe.predict_proba(X_val)[:, idx_min]

    thresholds = np.linspace(0, 1, 201)
    candidates = []
    for thr in thresholds:
        y_pred = np.where(proba_min >= thr, minority_label, 1.0)
        rec_min = recall_score(y_val, y_pred, pos_label=minority_label, zero_division=0)
        prec_min = precision_score(y_val, y_pred, pos_label=minority_label, zero_division=0)
        f1_min = f1_score(y_val, y_pred, pos_label=minority_label, zero_division=0)
        candidates.append((thr, rec_min, prec_min, f1_min))

    # Filtrar por meta de recall
    feasible = [c for c in candidates if c[1] >= recall_target]
    if not feasible:
        # Se ninguém bate a meta, pega o maior recall possível
        feasible = sorted(candidates, key=lambda x: x[1], reverse=True)[:1]

    if preference == 'precision':
        best = max(feasible, key=lambda x: (x[2], x[1]))  # prioriza precisão, desempata por recall
    else:
        best = max(feasible, key=lambda x: (x[3], x[1]))  # prioriza F1, desempata por recall

    thr, rec_min, prec_min, f1_min = best
    return thr, {'rec_min': rec_min, 'prec_min': prec_min, 'f1_min': f1_min}

def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_minority': recall_score(y_true, y_pred, pos_label=minority_label, zero_division=0),
        'precision_minority': precision_score(y_true, y_pred, pos_label=minority_label, zero_division=0),
        'f1_minority': f1_score(y_true, y_pred, pos_label=minority_label, zero_division=0),
    }

# Carregar dados e splits
df = pd.read_csv(CSV_PATH)
y = df[TARGET]
X = df.drop(columns=[TARGET])

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=random_state_global, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=1/3, random_state=random_state_global, stratify=y_temp)
print(f"Formas: treino={X_train.shape}, teste={X_test.shape}, val={X_val.shape}")

# Rodar por momento
rows = []
for m in range(1, 6):
    cols = [c for c in MOMENTS[m] if c in X.columns]
    # Tuning de profundidade por F1 da minoria
    pipe, best_depth, val_f1_min = tune_depth_for_moment(X_train, y_train, X_val, y_val, cols, candidate_depths)

    # Escolha de threshold com meta de recall mínima por momento
    thr, val_stats = choose_threshold_with_recall_target(pipe, X_val[cols], y_val, RECALL_MIN_TARGET[m], preference='precision')

    # Avaliação no teste
    y_pred_test = predict_with_threshold(pipe, X_test[cols], thr)
    metrics = compute_metrics(y_test, y_pred_test)

    # Print opcional detalhado
    print(f"\nMomento {m} (cols={len(cols)}) | max_depth={best_depth} | thr={thr:.3f} | meta_recall={RECALL_MIN_TARGET[m]:.2f}")
    print(f"Val (threshold): recall_min={val_stats['rec_min']:.3f}, precision_min={val_stats['prec_min']:.3f}, f1_min={val_stats['f1_min']:.3f}")
    print(f"Teste: Acc={metrics['accuracy']:.4f}, Recall_min={metrics['recall_minority']:.4f}, Prec_min={metrics['precision_minority']:.4f}, F1_min={metrics['f1_minority']:.4f}")
    print("Relatório por classe (teste):")
    print(classification_report(y_test, y_pred_test, digits=4, zero_division=0))
    print("Matriz de confusão (teste):")
    print(confusion_matrix(y_test, y_pred_test))

    rows.append({
        'moment': m,
        'n_cols': len(cols),
        'best_depth': best_depth,
        'thr': thr,
        'val_f1_min': val_f1_min,
        'val_recall_min_thr': val_stats['rec_min'],
        'val_precision_min_thr': val_stats['prec_min'],
        'val_f1_min_thr': val_stats['f1_min'],
        **metrics
    })

df_summary = pd.DataFrame(rows).sort_values('moment')
print("\nResumo consolidado:")
print(df_summary[['moment','n_cols','best_depth','thr','val_f1_min','val_recall_min_thr','val_precision_min_thr','val_f1_min_thr',
                  'accuracy','recall_macro','recall_minority','precision_minority','f1_minority']])

Formas: treino=(962, 20), teste=(275, 20), val=(138, 20)

Momento 1 (cols=3) | max_depth=None | thr=0.425 | meta_recall=0.50
Val (threshold): recall_min=0.750, precision_min=0.070, f1_min=0.128
Teste: Acc=0.4691, Recall_min=0.7500, Prec_min=0.0779, F1_min=0.1412
Relatório por classe (teste):
              precision    recall  f1-score   support

         0.0     0.0779    0.7500    0.1412        16
         1.0     0.9669    0.4517    0.6158       259

    accuracy                         0.4691       275
   macro avg     0.5224    0.6009    0.3785       275
weighted avg     0.9152    0.4691    0.5882       275

Matriz de confusão (teste):
[[ 12   4]
 [142 117]]

Momento 2 (cols=7) | max_depth=4 | thr=0.570 | meta_recall=0.45
Val (threshold): recall_min=0.500, precision_min=0.250, f1_min=0.333
Teste: Acc=0.8582, Recall_min=0.3750, Prec_min=0.1714, F1_min=0.2353
Relatório por classe (teste):
              precision    recall  f1-score   support

         0.0     0.1714    0.3750    0.23